## 📋 Overview

This notebook demonstrates comprehensive document processing across **multiple file formats** using NVIDIA's NV-Ingest pipeline integrated with Teradata Vector Store.

### Supported File Formats:
- 📄 **Text Files** (.txt) - Plain text documents
- 📝 **Markdown** (.md) - Formatted documentation
- 🌐 **HTML** (.html) - Web content and documentation
- ⚙️ **JSON** (.json) - Configuration and structured data
- 🖥️ **Shell Scripts** (.sh) - Deployment and automation scripts
- 📊 **PowerPoint** (.pptx) - Presentations
- 📝 **Word Documents** (.docx) - Text documents

### What You'll Learn:
- 🔄 Process diverse document types in a unified pipeline
- 💾 Store multiformat embeddings in Teradata Vector Store
- 🔍 Perform cross-format semantic searches
- 🤖 Build a RAG system that understands content from all formats

### Use Cases:
- Enterprise knowledge bases with mixed content types
- Technical documentation search
- Code repository analysis with documentation
- Configuration management and policy documents

## 📦 Import Required Libraries

Import all necessary modules for Teradata connectivity, NV-Ingest processing, and LangChain integration.

In [1]:
# Required imports
import teradatagenai
from teradataml import create_context, set_auth_token
from teradatagenai import (
    nvingest_retrieval, 
    create_nvingest_schema, 
    write_to_nvingest_vector_store, 
    TeradataVDB, 
    VectorStore, 
    TeradataAI
)
from langchain_teradata import TeradataVectorStore
from nv_ingest_client.client import Ingestor, NvIngestClient

from getpass import getpass
import os
from pathlib import Path
import glob

## 📂 Sample Data Files

This notebook uses sample enterprise data files located in the `sample_data/` directory:

### Files Included:
1. **product_catalog.txt** - Product information and pricing (Plain text)
2. **company_overview.md** - Company history and mission (Markdown)
3. **technical_documentation.html** - API documentation (HTML)
4. **configuration.json** - Application configuration settings (JSON)
5. **deployment_script.sh** - DevOps automation script (Shell script)
6. **presentation.pptx** - Company presentation slides (PowerPoint)
7. **documentation.docx** - Technical documentation (Word document)

**Note**: This notebook demonstrates processing various document formats including text files (.txt), PowerPoint presentations (.pptx), and Word documents (.docx) alongside structured formats like JSON, HTML, and Markdown.

## 🔗 Database Connection

Establish connection to Teradata Vantage and configure the environment.

In [39]:
# Configuration parameters
td_vs_name = "td_multiformat_docs"
sample_data_dir = "sample_data"

# Gather all sample files
file_patterns = [
    "*.txt",
    "*.md",
    "*.html",
    "*.json",
    "*.sh",
    "*.pptx",
    "*.docx"
]

# Collect all files
all_files = []
for pattern in file_patterns:
    files = glob.glob(f"{sample_data_dir}/{pattern}")
    all_files.extend(files)

print(f"📂 Found {len(all_files)} files to process:")
for f in all_files:
    print(f"   - {f}")

📂 Found 7 files to process:
   - sample_data\product_catalog.txt
   - sample_data\company_overview.md
   - sample_data\technical_documentation.html
   - sample_data\configuration.json
   - sample_data\deployment_script.sh
   - sample_data\TechVision_Overview.pptx
   - sample_data\TechVision_Report.docx


In [ ]:
# Store configuration - Get credentials securely
print("🔐 Teradata Connection Configuration")
print("=" * 60)

# Store configuration
os.environ['TD_HOST'] = getpass(prompt='hostname: ')
os.environ['TD_USERNAME'] = getpass(prompt='username: ')
os.environ['TD_PASSWORD'] = getpass(prompt='password: ')
os.environ['TD_BASE_URL'] = getpass(prompt='base_url: ')
os.environ['NV_INGEST_URL'] = getpass("Enter NV-Ingest host URL: ")
os.environ['TD_AUTH_MECH'] = 'BASIC'

# Create Teradata context
create_context()
print("\n✅ Connected to Teradata Vantage")

🔐 Teradata Connection Configuration

✅ Connected to Teradata Vantage

✅ Connected to Teradata Vantage


## 🚀 End-to-End Multi-Format Processing Pipeline

Process all document types in a single unified pipeline. NV-Ingest will intelligently:
- Extract text from all formats
- Parse structured data (JSON)
- Handle code and scripts
- Process HTML and Markdown
- Generate embeddings for semantic search
- Store everything in Teradata Vector Store

### Pipeline Features:
- **Unified Processing**: All formats processed together
- **Smart Extraction**: Format-aware content extraction
- **HNSW Search**: Fast similarity search with HNSW algorithm
- **Enterprise Scale**: Teradata's robust vector database

In [5]:
# Create TeradataVDB instance for multi-format processing
teradatavdb = TeradataVDB(
    name=td_vs_name,
    recreate=True,
    embeddings_dims=2048,
    search_algorithm="HNSW"
)

print(f"🗄️ Created Teradata Vector Store: {td_vs_name}")
print(f"📊 Embedding dimensions: 2048")
print(f"🔍 Search algorithm: HNSW")

🗄️ Created Teradata Vector Store: td_multiformat_docs
📊 Embedding dimensions: 2048
🔍 Search algorithm: HNSW


In [ ]:
# Process all files through NV-Ingest pipeline
print(f"\n🔄 Processing {len(all_files)} files through NV-Ingest...")
print("=" * 60)

ingestor = (
    Ingestor(message_client_hostname=os.environ['NV_INGEST_URL'], message_client_port=443)
    .files(all_files)  # Process all collected files
    .extract(
        extract_text=True,      # Extract text from all formats
        extract_tables=True,    # Extract tables from TXT, HTML, etc.
        extract_charts=False,   # Charts not applicable for most formats
        extract_images=False    # No images in these text-based formats
    )
    .embed()  # Generate embeddings
    .vdb_upload(vdb_op=teradatavdb)  # Upload to Teradata Vector Store
)

results = ingestor.ingest(batch_size=20)

print("\n" + "=" * 60)
print(f"✅ Successfully processed {len(results)} chunks")
print(f"💾 All content stored in Teradata Vector Store: {td_vs_name}")


🔄 Processing 8 files through NV-Ingest...


Error extracting content from sample_data\sales_data.csv: Failed to determine file type for: sample_data\sales_data.csv


Database connection established in 1137.47 milliseconds.


c:\Users\sp255175\Documents\GIT-REPO-TD\DartLite-New\DartLite\feature-store-oct\Lib\site-packages\teradatagenai\vector_store\vector_store.py:1038: UserWarning: Vector Store td_multiformat_docs does not exist. Call create() to create the same.
  warnings.warn(f"Vector Store {self.name} does not exist. Call create() to create the same.")
c:\Users\sp255175\Documents\GIT-REPO-TD\DartLite-New\DartLite\feature-store-oct\Lib\site-packages\teradatagenai\vector_store\vector_store.py:1038: UserWarning: Vector Store td_multiformat_docs does not exist. Call create() to create the same.
  warnings.warn(f"Vector Store {self.name} does not exist. Call create() to create the same.")


Vector store td_multiformat_docs creation started.
Use the 'status()' api to check the status of the operation.
Vector Store 'td_multiformat_docs' creation status:                vs_name                       status  retry_after
0  td_multiformat_docs  CREATING (GENERATING INDEX)           60


Purge requested, but save_to_disk was not configured. No files to purge.


Vector Store 'td_multiformat_docs' creation status:                vs_name status
0  td_multiformat_docs  READY

✅ Successfully processed 7 chunks
💾 All content stored in Teradata Vector Store: td_multiformat_docs


## 🔍 Cross-Format Semantic Search

Now let's search across all document formats using natural language queries. The vector store will find relevant content regardless of the source format.

In [ ]:
# Get NVIDIA API key for embeddings
print("🔑 NVIDIA API Configuration")
os.environ['NVIDIA_API_KEY'] = getpass(prompt='Enter NVIDIA API Key: ')

🔑 NVIDIA API Configuration
✅ API key configured


### Test Query 1: Product Information
Search for product details from text files.

In [ ]:
# Query about products from product catalog text file
query1 = "What products and services are offered in the product catalog?"

print(f"🔍 Query: {query1}")
print("=" * 60)

results1 = nvingest_retrieval(
    name=td_vs_name,
    queries=[query1],
    top_k=3,
    nvidia_api_key=os.environ['NVIDIA_API_KEY']
)

results1

🔍 Query: What products and services are offered in the product catalog?
Vector store td_multiformat_docs is initialized.
Vector store td_multiformat_docs is initialized.


[similar_objects_count:3
 similar_objects:
    TDID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

### Test Query 2: Company Information
Search for company details from Markdown documentation.

In [ ]:
# Query about company history
query2 = "Tell me about the company's history and mission"

print(f"🔍 Query: {query2}")
print("=" * 60)

results2 = nvingest_retrieval(
    name=td_vs_name,
    queries=[query2],
    top_k=3,
    nvidia_api_key=os.environ['NVIDIA_API_KEY']
)

results2

🔍 Query: Tell me about the company's history and mission
Vector store td_multiformat_docs is initialized.


[similar_objects_count:3
 similar_objects:
    TDID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

### Test Query 3: Technical Documentation
Search API documentation from HTML files.

In [ ]:
# Query about API documentation
query3 = "How do I authenticate with the API? What are the rate limits?"

print(f"🔍 Query: {query3}")
print("=" * 60)

results3 = nvingest_retrieval(
    name=td_vs_name,
    queries=[query3],
    top_k=3,
    nvidia_api_key=os.environ['NVIDIA_API_KEY']
)

results3

🔍 Query: How do I authenticate with the API? What are the rate limits?
Vector store td_multiformat_docs is initialized.


[similar_objects_count:3
 similar_objects:
    TDID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

### Test Query 4: Configuration Details
Search configuration from JSON files.

In [ ]:
# Query about configuration
query4 = "What cloud providers are configured and what security features are enabled?"

print(f"🔍 Query: {query4}")
print("=" * 60)

results4 = nvingest_retrieval(
    name=td_vs_name,
    queries=[query4],
    top_k=3,
    nvidia_api_key=os.environ['NVIDIA_API_KEY']
)

results4

🔍 Query: What cloud providers are configured and what security features are enabled?
Vector store td_multiformat_docs is initialized.


[similar_objects_count:3
 similar_objects:
    TDID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

### Test Query 5: Deployment Scripts
Search deployment procedures from shell scripts.

In [ ]:
# Query about deployment
query5 = "What are the steps in the deployment process and what health checks are performed?"

print(f"🔍 Query: {query5}")
print("=" * 60)

results5 = nvingest_retrieval(
    name=td_vs_name,
    queries=[query5],
    top_k=3,
    nvidia_api_key=os.environ['NVIDIA_API_KEY']
)

results5

🔍 Query: What are the steps in the deployment process and what health checks are performed?
Vector store td_multiformat_docs is initialized.


[similar_objects_count:3
 similar_objects:
    TDID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

### Test Query 6: PowerPoint Presentation Content
Search for information from PowerPoint presentations.

In [ ]:
# Query about presentation content from .pptx files
query6 = "What are the key topics and main points covered in the PowerPoint presentation?"

print(f"🔍 Query: {query6}")
print("=" * 60)

results6 = nvingest_retrieval(
    name=td_vs_name,
    queries=[query6],
    top_k=3,
    nvidia_api_key=os.environ['NVIDIA_API_KEY']
)

results6

🔍 Query: What are the key topics and main points covered in the PowerPoint presentation?
Vector store td_multiformat_docs is initialized.
Vector store td_multiformat_docs is initialized.


[similar_objects_count:3
 similar_objects:
    TDID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

### Test Query 7: Word Document Content
Search for information from Word documents.

In [ ]:
# Query about Word document content from .docx files
query7 = "What information is contained in the Word document? Summarize the main content."

print(f"🔍 Query: {query7}")
print("=" * 60)

results7 = nvingest_retrieval(
    name=td_vs_name,
    queries=[query7],
    top_k=3,
    nvidia_api_key=os.environ['NVIDIA_API_KEY']
)

results7

🔍 Query: What information is contained in the Word document? Summarize the main content.
Vector store td_multiformat_docs is initialized.
Vector store td_multiformat_docs is initialized.


[similar_objects_count:3
 similar_objects:
    TDID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

## 🔧 Advanced: Custom Embedding Configuration

Use specific embedding models and endpoints for more precise results.

In [ ]:
# Advanced retrieval with custom embeddings
print("⚙️ Advanced Embedding Configuration")
embedding_endpoint = getpass(prompt='NV INGEST Embeddings URL')
model_name = 'nvidia/llama-3.2-nv-embedqa-1b-v2'

advanced_query = "Provide a comprehensive overview of all the company's products, services, and technical capabilities"

print(f"\n🔍 Advanced Query: {advanced_query}")
print("=" * 60)

advanced_results = nvingest_retrieval(
    name=td_vs_name,
    embedding_endpoint=embedding_endpoint,
    model_name=model_name,
    nvidia_api_key=os.environ['NVIDIA_API_KEY'],
    queries=[advanced_query],
    top_k=5
)

advanced_results

⚙️ Advanced Embedding Configuration

🔍 Advanced Query: Provide a comprehensive overview of all the company's products, services, and technical capabilities
Vector store td_multiformat_docs is initialized.
Vector store td_multiformat_docs is initialized.


c:\Users\sp255175\Documents\GIT-REPO-TD\DartLite-New\DartLite\feature-store-oct\Lib\site-packages\llama_index\embeddings\nvidia\base.py:170: UserWarning: Expected format is 'http://host:port'. Rest is Ignored.
  warnings.warn(f"{expected_format} Rest is Ignored.")


[similar_objects_count:5
 similar_objects:
    TDID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

## 🤖 Building a Multi-Format RAG Pipeline

Create a complete RAG system using LangChain that can intelligently answer questions by retrieving relevant content from any document format.

In [24]:
# Create TeradataVectorStore instance for RAG
vs = TeradataVectorStore(td_vs_name)
retriever = vs.as_retriever(search_kwargs={"top_k": 5})

print(f"✅ Vector store retriever created for: {td_vs_name}")

Vector store td_multiformat_docs is initialized.
✅ Vector store retriever created for: td_multiformat_docs
✅ Vector store retriever created for: td_multiformat_docs


In [ ]:
# Update vector store with embedding configuration
vs.update(
    embeddings_model=model_name,
    embeddings_dims=2048,
    embeddings_base_url=getpass(prompt='Enter Embeddings Base URL: ')
)

print("✅ Vector store configuration updated")

Vector store td_multiformat_docs update started.
Use the 'status()' api to check the status of the operation.
✅ Vector store configuration updated


In [27]:
# Check vector store status
vs.status()

,vs_name,status
0,td_multiformat_docs,READY


In [ ]:
# Import LangChain components for RAG
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.chat_models import init_chat_model

print("🔑 LLM Configuration - AWS Bedrock")
print("=" * 60)

# Setup AWS Configuration
aws_access_key = getpass(prompt='Enter AWS Access Key ID: ')
aws_secret_key = getpass(prompt='Enter AWS Secret Access Key: ')
aws_region = getpass(prompt='Enter AWS Region: ')

# Initialize Bedrock LLM (Claude)
# llm = init_chat_model(
#     "ARN-DETAILS",
#     model_provider="bedrock_converse",
#     provider="anthropic", 
#     region_name=aws_region,
#     aws_access_key_id=aws_access_key,
#     aws_secret_access_key=aws_secret_key
# )

llm = init_chat_model(
    model_name = model_name,
    model_provider="bedrock_converse",
    region_name=aws_region,
    aws_access_key_id=aws_access_key,
    aws_secret_access_key=aws_secret_key
)

print("\n✅ LLM initialized : AWS Bedrock ")

🔑 LLM Configuration - AWS Bedrock

✅ LLM initialized: AWS Bedrock (Claude 3.5 Sonnet)

✅ LLM initialized: AWS Bedrock (Claude 3.5 Sonnet)


In [29]:
# Create comprehensive prompt template for multi-format content
prompt = PromptTemplate.from_template(
    """You are an AI assistant analyzing enterprise documentation from multiple sources including:
- Text files (product catalogs)
- PowerPoint presentations (company presentations)
- Word documents (technical documentation)
- Markdown files (company documentation)
- HTML files (technical documentation)
- JSON files (configuration settings)
- Shell scripts (deployment procedures)

Use the following retrieved context to provide a comprehensive, accurate answer.
When relevant, mention which type of document the information comes from.

Context:
{context}

Question: {question}

Answer (be specific and cite sources when helpful):"""
)

# Build the RAG chain
rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG chain built successfully")
print("=" * 60)
print("🎉 Multi-Format RAG Pipeline Ready!")
print("   You can now ask questions about any content in your documents.\n")

✅ RAG chain built successfully
🎉 Multi-Format RAG Pipeline Ready!
   You can now ask questions about any content in your documents.



## 💬 Ask Questions Across All Document Types

Now you can ask questions and the RAG system will intelligently search across all document formats to provide comprehensive answers.

### Example 1: Product and Business Intelligence

In [30]:
# Business intelligence query
print("💼 Business Intelligence Query")
print("=" * 60)

question1 = """Provide a comprehensive overview of the company's product portfolio, 
including pricing, sales performance, and target markets. Which product is performing best?"""

print(f"Q: {question1}\n")

response1 = rag_chain.invoke(question1)

print("=" * 60)
print(f"A: {response1}")

💼 Business Intelligence Query
Q: Provide a comprehensive overview of the company's product portfolio, 
including pricing, sales performance, and target markets. Which product is performing best?

A: # Comprehensive Product Portfolio Overview - TechVision Solutions

Based on the enterprise documentation available, here's a detailed analysis of TechVision Solutions' product portfolio:

## Product Lineup (from Product Catalog 2024)

### 1. **CloudSync Pro**
- **Description**: Advanced cloud synchronization platform for enterprise data management
- **Pricing**: $4,999/year
- **Key Features**: Real-time sync, 256-bit encryption, multi-cloud support, automated backups
- **Target Market**: Large enterprises, Fortune 500 companies
- **Launch**: 2019 (first major product)

### 2. **DataGuard Analytics**
- **Description**: AI-powered data security and analytics platform
- **Pricing**: $7,500/year (highest-priced product)
- **Key Features**: Machine learning threat detection, compliance monitorin

### Example 2: Technical Integration Question

In [31]:
# Technical documentation query
print("\n🔧 Technical Integration Query")
print("=" * 60)

question2 = """I want to integrate with the CloudSync Pro API. 
How do I authenticate, what are the rate limits, and which cloud providers are supported?"""

print(f"Q: {question2}\n")

response2 = rag_chain.invoke(question2)

print("=" * 60)
print(f"A: {response2}")


🔧 Technical Integration Query
Q: I want to integrate with the CloudSync Pro API. 
How do I authenticate, what are the rate limits, and which cloud providers are supported?

A: # CloudSync Pro API Integration Guide

Based on the documentation, here's what you need to know:

## Authentication

**From the API Documentation (Markdown):**
- All API requests require authentication using API keys
- Include your API key in the request header as: `Authorization: Bearer YOUR_API_KEY`

**To obtain an API key:**
1. Log in to your CloudSync Pro dashboard
2. Navigate to Settings > API Keys
3. Click "Generate New Key"
4. Copy and securely store your API key

## Rate Limits

**From the API Documentation (Markdown table):**

The rate limits vary by subscription tier:

| Tier         | Requests per Minute | Requests per Day |
|--------------|---------------------|------------------|
| **Free**     | 60                  | 5,000            |
| **Professional** | 300             | 50,000           |
| **E

### Example 3: DevOps and Deployment

In [32]:
# DevOps query
print("\n🚀 DevOps Query")
print("=" * 60)

question3 = """Explain the deployment process for CloudSync Pro. 
What prerequisites are checked, what health checks are performed, and what happens if deployment fails?"""

print(f"Q: {question3}\n")

response3 = rag_chain.invoke(question3)

print("=" * 60)
print(f"A: {response3}")


🚀 DevOps Query
Q: Explain the deployment process for CloudSync Pro. 
What prerequisites are checked, what health checks are performed, and what happens if deployment fails?

A: # CloudSync Pro Deployment Process

Based on the **deployment shell script** provided, here's a comprehensive overview of the CloudSync Pro deployment process:

## Basic Deployment Information

**Application:** CloudSync Pro  
**Current Version:** 3.2.1  
**Maintained by:** DevOps Team - TechVision Solutions  
**Deployment Environment:** Production (default), can be specified as parameter  

## Key Configuration Details

From the **shell script**, the deployment uses:
- **Script Version:** 3.2.1
- **Deployment Environment:** Configurable via command-line parameter (defaults to "production")
- **Logging:** All deployment activities are logged to `/var/log/deployments/${APP_NAME}_${TIMESTAMP}.log`
- **Error Handling:** The script uses `set -e` (exit on error) and `set -u` (exit on undefined variable) for robust e

### Example 4: Company and Strategic Information

In [33]:
# Strategic overview query
print("\n🏢 Strategic Overview Query")
print("=" * 60)

question4 = """Tell me about TechVision Solutions' company history, mission, 
global presence, and recent achievements. How has the company evolved?"""

print(f"Q: {question4}\n")

response4 = rag_chain.invoke(question4)

print("=" * 60)
print(f"A: {response4}")


🏢 Strategic Overview Query
Q: Tell me about TechVision Solutions' company history, mission, 
global presence, and recent achievements. How has the company evolved?

A: # TechVision Solutions: Comprehensive Company Overview

## Mission Statement
According to the company documentation (Markdown file), TechVision Solutions' mission is:
> "Empowering businesses through innovative technology solutions that drive digital transformation and operational excellence."

## Company Evolution and History

TechVision Solutions has experienced remarkable growth since its inception. Based on the company overview documents, here's their trajectory:

### Timeline of Growth:
- **2018**: Company founded with just 5 employees
- **2019**: Launched their first major product, CloudSync Pro
- **2020**: Expanded into the European market and grew to 50 employees
- **2021**: Introduced DataGuard Analytics and secured Series B funding
- **2022**: Opened Asia Pacific office with 200+ employees
- **2023**: Launched

### Example 5: Configuration and Security

In [34]:
# Configuration and security query
print("\n🔒 Security & Configuration Query")
print("=" * 60)

question5 = """What security features are implemented in CloudSync Pro? 
Include details about encryption, authentication, and monitoring."""

print(f"Q: {question5}\n")

response5 = rag_chain.invoke(question5)

print("=" * 60)
print(f"A: {response5}")


🔒 Security & Configuration Query
Q: What security features are implemented in CloudSync Pro? 
Include details about encryption, authentication, and monitoring.

A: # CloudSync Pro Security Features

Based on the enterprise documentation, CloudSync Pro implements multiple layers of security features:

## Encryption

**From the Product Catalog (Text file):**
- **256-bit encryption** is a core feature of CloudSync Pro

**From the Configuration Settings (JSON file):**
- **AES256 bucket encryption** is enabled for AWS S3 storage
- **SSL/TLS encryption** is enabled for database connections (`"ssl_enabled": true`)
- **Versioning** is enabled on cloud storage, which helps protect against accidental deletion or malicious modification

## Authentication

**From the API Documentation (Markdown file):**
- **API key-based authentication** is required for all API requests
- API keys must be included in the request header: `Authorization: Bearer YOUR_API_KEY`
- API keys can be generated through the 

### Example 6: Cross-Domain Analysis

In [35]:
# Comprehensive cross-domain query
print("\n🎯 Comprehensive Cross-Domain Query")
print("=" * 60)

question6 = """As a potential enterprise customer, give me a complete picture: 
What does TechVision offer, how reliable are their products based on deployment processes, 
what security measures are in place, and how is the company performing financially?"""

print(f"Q: {question6}\n")

response6 = rag_chain.invoke(question6)

print("=" * 60)
print(f"A: {response6}")


🎯 Comprehensive Cross-Domain Query
Q: As a potential enterprise customer, give me a complete picture: 
What does TechVision offer, how reliable are their products based on deployment processes, 
what security measures are in place, and how is the company performing financially?

A: # TechVision Solutions - Complete Enterprise Assessment

Based on the available documentation, here's a comprehensive overview for potential enterprise customers:

## Company Profile & Credibility

**Company Background** (from company documentation):
- Founded in 2018 with just 5 employees
- Rapid growth trajectory: now serving 1000+ customers (as of 2024)
- Global presence with offices in North America, Europe, and Asia Pacific
- 500+ enterprise customers as of 2023
- Industry recognition includes **Gartner Cool Vendor** and **Forbes Cloud 100 Rising Star** awards (from PowerPoint presentation)

**Mission**: Empowering businesses through innovative technology solutions that drive digital transformation and

## 📊 Inspect Vector Store Contents

Let's examine what's stored in our vector database and see how content from different formats is represented.

In [36]:
# View stored embeddings
vs_inspect = VectorStore(name=td_vs_name)
print(f"📊 Vector Store Contents: {td_vs_name}")
print("=" * 60)
vs_inspect.get_indexes_embeddings().head(20)

Vector store td_multiformat_docs is initialized.
📊 Vector Store Contents: td_multiformat_docs
📊 Vector Store Contents: td_multiformat_docs


DataBaseName                                TableName  TDID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

In [38]:
# Get statistics about the vector store
embeddings_df = vs_inspect.get_indexes_embeddings()
print(f"\n📈 Vector Store Statistics")
print("=" * 60)

# Convert to pandas to get row count
embeddings_pd = embeddings_df.to_pandas()
total_chunks = len(embeddings_pd)

print(f"Total chunks stored: {total_chunks}")
print(f"Embedding dimensions: {embeddings_pd.shape[1] if total_chunks > 0 else 'N/A'}")
print(f"Vector store name: {td_vs_name}")


📈 Vector Store Statistics
Total chunks stored: 9
Embedding dimensions: 8
Vector store name: td_multiformat_docs
Total chunks stored: 9
Embedding dimensions: 8
Vector store name: td_multiformat_docs


## 🧹 Cleanup (Optional)

Run this cell if you want to remove the vector store and clean up resources.

In [ ]:
# Uncomment to destroy the vector store
vs.destroy()
print(f"🗑️ Vector store '{td_vs_name}' destroyed")

## 🎉 Summary

Congratulations! You've successfully:

1. ✅ **Processed multiple file formats** in a unified pipeline:
   - Text files (.txt) - Product catalogs
   - PowerPoint presentations (.pptx) - Company presentations
   - Word documents (.docx) - Technical documentation
   - Markdown documentation (.md) - Company information
   - HTML web content (.html) - API documentation
   - JSON configuration (.json) - Settings and config
   - Shell scripts (.sh) - Deployment automation

2. ✅ **Created a unified vector store** in Teradata with all content

3. ✅ **Performed cross-format semantic searches** finding relevant content regardless of source format

4. ✅ **Built a comprehensive RAG pipeline** that answers questions by intelligently retrieving from any document type

5. ✅ **Demonstrated real-world use cases** from business intelligence to technical documentation

### Key Capabilities Demonstrated:

#### 📄 Multi-Format Support:
- Unified processing of diverse document types
- Format-aware content extraction
- Consistent embedding generation across formats

#### 🔍 Intelligent Search:
- Semantic search across all document types
- Context-aware retrieval
- Cross-format information synthesis

#### 🤖 Enterprise RAG:
- LLM-powered question answering
- Multi-source context integration
- Production-ready architecture

### Best Practices:

- **Organize files**: Use consistent naming and folder structures
- **Quality content**: Ensure source documents are well-formatted
- **Regular updates**: Re-process documents when content changes
- **Monitor performance**: Track search accuracy and relevance
- **Security**: Implement proper access controls for sensitive documents

### Resources:
- [NVIDIA NV-Ingest Documentation](https://docs.nvidia.com/nv-ingest/)
- [Teradata Vector Store Guide](https://docs.teradata.com/)
- [LangChain RAG Documentation](https://python.langchain.com/docs/use_cases/question_answering/)
- [Teradata GenAI Package](https://github.com/Teradata/teradatagenai)

---

The same RAG system will seamlessly incorporate content from these additional formats!